# RQ5: What is the impact of the labeled data fraction on the performance of different encoders?

## Define base configs

In [ ]:
import pandas as pd
metric_used = 'accuracy'
apply_correction_factor = True


def extract_metric_target(df):
    metric_map = {
        "har": "accuracy",
        "hapt": "accuracy",
    }
    return df["pipeline/task"].map(metric_map)


def extract_metric(df):
    metric_results = []
    metric_map = {
        "har": "metric/classification/accuracy",
        "hapt": "metric/classification/accuracy",
    }

    for _, row in df.iterrows():
        metric_column = metric_map[row["pipeline/task"]]
        metric_value = row[metric_column]
        metric_results.append(metric_value)

    return metric_results

In [ ]:
from pathlib import Path


from utils import (
    calculate_variant_wilcoxon,
    create_precedence_graph,
    prepare_experiment_df,
    plot_comparison_bar
)

# disable warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Path to the experiment results (parsed) summarized_executions_fixed
filename = 'clean_saved_metrics_paper_final'
summarized_executions_path = Path(f"{filename}.csv")

# summarized_executions_path = Path("summarized_executions_fixed.csv")

# Location to save figures and tables
figures_path = Path("results/figures/")
tables_path = Path("results/tables/")
figures_path.mkdir(parents=True, exist_ok=True)
tables_path.mkdir(parents=True, exist_ok=True)
print(f"Sucessfully created directories '{figures_path}' and '{tables_path}'")

## Utility functions

First, we define some utility functions that will be used to parse the benchmarks results and to generate the plots.

In [ ]:
def result_pairwise_wilcoxon(
    df, variants_variables, filters={}, show_stats=False,apply_bonferroni= False,
):
    df = prepare_experiment_df(
        df,
        **filters,
        verbose=True,
    )
    # display(df)
    if show_stats:
        print("Dataframe has", len(df.index), "rows")
        for col in df.columns:
            values = df[col].unique()
            print(f" - {col} ({len(values)})", values)

    return calculate_variant_wilcoxon(df, variants_variables, threshold=0.05,apply_bonferroni= apply_bonferroni)


def result_precedence_graph(
    df, variants_variables, filters={}, show_stdev=False,apply_bonferroni= False,show_df_tests=False
):
    w_df = result_pairwise_wilcoxon(df, variants_variables, filters,apply_bonferroni= apply_bonferroni,)
    if show_df_tests:
        print("Pairwise Wilcoxon test results:")
        display(w_df)
    return create_precedence_graph(w_df, show_stdev=show_stdev,apply_bonferroni= apply_bonferroni),w_df


def show_precedence_graph(
    df, variants_variables, filters, filename_suffix=None, show_stdev=False,apply_bonferroni= False,show_df_tests=False,filename=None
):

    dot,w_df = result_precedence_graph(
        df=df,
        variants_variables=variants_variables,
        filters=filters,
        show_stdev=show_stdev,
        apply_bonferroni=apply_bonferroni,
        show_df_tests=show_df_tests,
    )
    if filename is None:
        
        filename = "precedence_graph-" + "-".join(variants_variables)
    if filename_suffix:
        filename = filename + "-" + filename_suffix
    dot.render(
        filename=filename, directory=figures_path, format="png", cleanup=True
    )
    print(f"Precedence graph saved to '{figures_path / filename}.png'\n")
    display(dot)
    return dot,w_df


def summarize_backbone_performance(df):
    # Get all unique backbones (from both Variant 1 and Variant 2)
    all_backbones = set(df["Variant 1"]).union(set(df["Variant 2"]))
    
    # Initialize a dictionary to store stats for each backbone
    backbone_stats = {}
    
    for backbone in all_backbones:
        # Get all rows where the backbone appears (either in Variant 1 or Variant 2)
        mask = (df["Variant 1"] == backbone) | (df["Variant 2"] == backbone)
        relevant_rows = df[mask]
        
        # Extract means and stds where the backbone is involved
        means = []
        stds = []
        
        for _, row in relevant_rows.iterrows():
            if row["Variant 1"] == backbone:
                means.append(row["Variant 1 Mean"])
                stds.append(row["Variant 1 stdev"])
            else:
                means.append(row["Variant 2 Mean"])
                stds.append(row["Variant 2 stdev"])
        
        # Compute mean and std across all occurrences
        mean_performance = np.mean(means) if means else 0
        avg_std = np.mean(stds) if stds else 0
        
        backbone_stats[backbone] = {
            "Mean": mean_performance,
            "Std": avg_std,
            "Mean ± Std": f"{np.round(mean_performance*100, 1)}% ± {np.round(avg_std*100, 1)}%"
        }
    # --- Original logic for wins/losses ---
    sig_df = df[df["Significant"] == True].copy()
    
    # Determine winner and loser based on means
    sig_df["Winner"] = sig_df.apply(
        lambda row: row["Variant 1"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 2"], 
        axis=1
    )
    sig_df["Loser"] = sig_df.apply(
        lambda row: row["Variant 2"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 1"], 
        axis=1
    )
    
    # Count wins and losses
    win_counts = sig_df["Winner"].value_counts()
    loss_counts = sig_df["Loser"].value_counts()
    
    # Create summary DataFrame
    summary_df = pd.DataFrame.from_dict(backbone_stats, orient="index")
    summary_df.index.name = "Backbone"
    summary_df.reset_index(inplace=True)
    
    # Ensure all backbones are included (even if no wins/losses)
    summary_df["Wins"] = summary_df["Backbone"].map(win_counts).fillna(0).astype(int)
    summary_df["Losses"] = summary_df["Backbone"].map(loss_counts).fillna(0).astype(int)
    summary_df["Net Score"] = summary_df["Wins"] - summary_df["Losses"]
    
    # Sort by Net Score (descending)
    summary_df.sort_values(["Net Score", "Mean"], ascending=[False, False], inplace=True)
    
    return summary_df


In [ ]:
def aggregate_backbone_performance(combined_df):
    """
    Processes a combined DataFrame of backbone results to produce:
    1. Technique-specific performance (mean ± std)
    2. Backbone totals across all techniques
    
    Args:
        combined_df: DataFrame containing results from multiple techniques
        
    Returns:
        tuple: (technique_summary_df, backbone_totals_df)
    """
    # --- Technique-Specific Summary ---
    technique_summary = combined_df.copy()
    
    # Convert to percentages if needed (assuming original means are 0-1)
    technique_summary["Mean"] = technique_summary["Mean"] * 100
    technique_summary["Std"] = technique_summary["Std"] * 100
    
    # Format performance string
    technique_summary["Performance"] = (
        technique_summary["Mean"].round(1).astype(str) + 
        "% ± " + 
        technique_summary["Std"].round(1).astype(str) + "%"
    )
    
    # Sort by Net Score then Mean
    technique_summary = technique_summary.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # --- Backbone Totals ---
    # Extract backbone name (before " + ")
    combined_df["Backbone Only"] = combined_df["Backbone"].str.split(" \+ ").str[0]
    
    # Group and aggregate
    backbone_totals = combined_df.groupby("Backbone Only").agg({
        "Wins": "sum",
        "Losses": "sum",
        "Net Score": "sum",
        "Mean": lambda x: np.mean(x) * 100,  # Convert to percentage
        "Std": lambda x: np.mean(x) * 100
    }).reset_index()
    
    # Format performance
    backbone_totals["Performance"] = (
        backbone_totals["Mean"].round(1).astype(str) + 
        "% ± " + 
        backbone_totals["Std"].round(1).astype(str) + "%"
    )
    
    # Clean up
    backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
    backbone_totals = backbone_totals.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # Special handling for TS2Vec if present
    if "TS2Vec" in backbone_totals["Backbone"].values:
        backbone_totals["Backbone"] = backbone_totals["Backbone"].replace({
            "TS2Vec": "TS2Vec (Partial)"
        })
    
    # Select final columns
    backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]
    
    return technique_summary, backbone_totals

def generate_performance_tables(df_list, technique_names=None):
    """
    Complete workflow from individual technique DataFrames to final tables
    
    Args:
        df_list: List of DataFrames for each technique
        technique_names: Optional list of technique names
        
    Returns:
        tuple: (combined_df, technique_summary, backbone_totals)
    """
    # Add technique identifiers if provided
    if technique_names and len(technique_names) == len(df_list):
        for df, name in zip(df_list, technique_names):
            df["Technique"] = name
    
    # Combine all DataFrames
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Generate summary tables
    technique_summary, backbone_totals = aggregate_backbone_performance(combined_df)
    
    return combined_df, technique_summary, backbone_totals



In [ ]:
summarized_executions_path

In [ ]:
df = pd.read_csv(summarized_executions_path)
df

In [ ]:
# sanity check

df_tnc = df[df['tsk_pretext'].str.contains('TNC')]
df_tnc

df_tnc = df_tnc[df_tnc['ft_strategy'].str.contains('Full Finetune')]

df_tnc = df_tnc[df_tnc['frac_dtarget'].astype(str) ==('1000.0')]

df_tnc = df_tnc[df_tnc['backbone'].str.contains('ResNet-SE-5')]

display(df_tnc)

# mean and std of sccuracy
mean_accuracy = df_tnc['metric'].mean()*100
std_accuracy = df_tnc['metric'].std()*100
print(f"Mean Accuracy: {mean_accuracy:.1f}")
print(f"Std Accuracy: {std_accuracy:.1f}")


In [ ]:
# sanity check

df_tnc = df[df['tsk_pretext'].str.contains('LFR')]
df_tnc

df_tnc = df_tnc[df_tnc['ft_strategy'].str.contains('Full Finetune')]

df_tnc = df_tnc[df_tnc['frac_dtarget'].astype(str) ==('1000.0')]

df_tnc = df_tnc[df_tnc['backbone'].str.contains('ResNet-SE-5')]

display(df_tnc)

# mean and std of sccuracy
mean_accuracy = df_tnc['metric'].mean()*100
std_accuracy = df_tnc['metric'].std()*100
print(f"Mean Accuracy: {mean_accuracy:.1f}")
print(f"Std Accuracy: {std_accuracy:.1f}")


In [ ]:
rows = []

for technique in sorted(df['tsk_pretext'].unique()):
    df_t = df[df['tsk_pretext'] == technique]
    df_t = df_t[df_t['ft_strategy'].str.contains('Full Finetune')]

    for backbone in sorted(df_t['backbone'].unique()):
        df_b = df_t[df_t['backbone'] == backbone]

        for frac in sorted(df_b['frac_dtarget'].unique()):
            df_f = df_b[df_b['frac_dtarget'] == frac]

            mean_acc = df_f['metric'].mean() * 100
            std_acc  = df_f['metric'].std() * 100
            n_runs   = df_f['metric'].count()

            rows.append({
                'tsk_pretext': technique,
                'backbone': backbone,
                'frac_dtarget': frac,
                'mean_acc': mean_acc,
                'std_acc': std_acc,
                'acc': f"{mean_acc:.1f} ± {std_acc:.1f}",
                'n': n_runs
            })
summary_loop = pd.DataFrame(rows)
display(summary_loop)


In [ ]:
sanity_table_loop = summary_loop.pivot_table(
    index=['tsk_pretext', 'backbone'],
    columns='frac_dtarget',
    values='acc',
    aggfunc='first'
).sort_index()

display(sanity_table_loop)


backbones cnn better but a lot of variance we need to take a close look on whats happening

### P5. Qual o melhor backbone geral para HAR de acordo com a porcentagem de dados em caso de Finetune?

Qual o melhor backbone para refino com  1% dos dados?
Qual o melhor backbone para refino com  5% dos dados?
Qual o melhor backbone para refino com  10% dos dados?
Qual o melhor backbone para refino com  50% dos dados?
Qual o melhor backbone para refino com  100% dos dados?



In [ ]:
import seaborn as sns
# Custom backbone order and Viridis palette
custom_palette = {
    "RNN": "tab:blue",
    "IMU Transformer": "tab:orange",
    "ResNet-1D": "tab:green",
    "CNN-PFF": "tab:red",
    'TS2Vec Encoder': 'tab:purple',
    'TS-TCC Encoder': 'tab:brown',
}
# Define the order of backbones
backbone_order = ["RNN", "IMU Transformer", "ResNet-1D","ResNet-SE-5", "CNN-PFF",'TS2Vec Encoder','TS-TCC Encoder']

# Get colors from Viridis
viridis_colors = sns.color_palette("deep", n_colors=len(backbone_order))
viridis_palette = {backbone: color for backbone, color in zip(backbone_order, viridis_colors)}

# Aesthetic settings
sns.set(style="whitegrid", font_scale=1.5)
custom_params = {
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",
    "grid.color": "black",
    "grid.linestyle": "--",
    "grid.linewidth": 1.5
}
sns.set_context("paper", rc=custom_params)

label_fontsize = 18
tick_fontsize = 16

In [ ]:


finetune_df = df[df["ft_strategy"] == "Full Finetune"]
freeze_df = df[df["ft_strategy"] == "Freeze"]

In [ ]:
finetune_df

In [ ]:
# Aggregate mean and std
mean_df = finetune_df.groupby(['backbone', 'frac_dtarget'])['metric'].agg(['mean', 'std']).reset_index()
# mean_df = mean_df.round({'mean': 1, 'std': 1})
mean_df

In [ ]:
mean_df = (
    finetune_df
    .groupby(['backbone', 'frac_dtarget'])['metric']
    .agg(['mean', 'std'])
    .reset_index()
)

# convert to percentage strings with 1 decimal place
mean_df['mean'] = (mean_df['mean'] * 100).round(1).astype(str) + '%'
mean_df['std']  = (mean_df['std']  * 100).round(1).astype(str) + '%'

mean_df

In [ ]:
mean_df = (
    finetune_df
    .groupby(['tsk_pretext', 'backbone', 'frac_dtarget'])['metric']
    .agg(['mean', 'std'])
    .reset_index()
)

# convert to percentage strings with 1 decimal place
mean_df['mean'] = (mean_df['mean'] * 100).round(1).astype(str) + '%'
mean_df['std']  = (mean_df['std']  * 100).round(1).astype(str) + '%'

mean_df

for tsk, df_sub in mean_df.groupby('tsk_pretext'):
    print(f"\n=== {tsk} ===")
    display(df_sub)



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a custom palette
custom_palette = {
    "RNN": "tab:blue",
    "IMU Transformer": "tab:orange",
    "ResNet-1D": "tab:green",
    "CNN-PFF": "tab:red",
    'TS2Vec Encoder': 'tab:purple',
    'TS-TCC Encoder': 'tab:brown',
    'ResNet-SE-5': 'tab:pink',
}

# Convert frac_dtarget to categorical and update to percentage strings
# try:
#     finetune_df["frac_dtarget"] = finetune_df["frac_dtarget"].astype(float) * 100
#     finetune_df["frac_dtarget"] = finetune_df["frac_dtarget"].astype(str) + '%'
# except ValueError as e:
#     print(f"Error converting frac_dtarget: {e}")

# Group by backbone and frac_dtarget, and calculate mean accuracy and standard deviation
mean_df = finetune_df.groupby(['backbone','frac_dtarget'])['metric'].agg(['mean', 'std']).reset_index()
display(mean_df)
mean_df["frac_dtarget"] = mean_df["frac_dtarget"].astype(str)
# Set enhanced style parameters
sns.set_style("whitegrid", rc={
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",  # Bold axis labels
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.titlesize": 14,        # Larger title (though we'll remove it)
})

# Create figure with larger dimensions
plt.figure(figsize=(11, 7), layout='constrained')

# Create pointplot with enhanced visibility
ax = sns.pointplot(
    data=mean_df, 
    x='frac_dtarget', 
    y='mean', 
    hue='backbone',
    # palette=viridis_palette,
    hue_order=["RNN", "IMU Transformer", "ResNet-1D",'ResNet-SE-5', "CNN-PFF",'TS2Vec Encoder','TS-TCC Encoder'],
    order=["1.0", "5.0","10.0","25.0","50.0","100.0","200.0","1000.0"],
    errwidth=2.0,      # Thicker error bars
    capsize=0.15,      # Slightly larger caps
    # markersize=10      # Larger points
)

# Enhanced axis labels
plt.xlabel("Number of Samples", 
          fontsize=16,  # Increased from 12
          fontweight='bold',
          labelpad=12)
plt.ylabel("Balanced Accuracy", 
          fontsize=16,  # Increased from 12
          fontweight='bold',
          labelpad=12)

# Bold axis ticks with larger font
plt.xticks(rotation=45, 
          fontsize=13,  # Increased from 11
          fontweight='bold')
plt.yticks(fontsize=13,  # Increased from 11
          fontweight='bold')

# Y-axis formatting
plt.ylim(0.3, 1.0)
ax.set_yticklabels(['{:.0f}%'.format(y*100) for y in ax.get_yticks()])

# Enhanced grid
plt.grid(True, alpha=0.7)

# Upgraded legend
handles, labels = ax.get_legend_handles_labels()
legend = plt.legend(
    handles, 
    labels, 
    title="Backbone",
    title_fontsize=18,  # Increased from 12
    fontsize=16,        # Increased from 11
    bbox_to_anchor=(1, 1),
    loc='upper left',
    frameon=True,
    framealpha=1,
    edgecolor='black'
)

# Make legend title bold
legend.get_title().set_fontweight('bold')

# Save high-quality output
plt.savefig('finetune_rq42.png', 
           dpi=350,      # Higher than standard 300
           bbox_inches='tight', 
           transparent=True)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a custom palette
# custom_palette = {
#     "RNN": "blue",
#     "IMU Transformer": "red",
#     "ResNet-1D": "green",
#     "CNN-PFF": "orange",
#     "TS Encoder": "purple"
# }

# Group by backbone and frac_dtarget, and calculate mean accuracy and standard deviation
mean_df = finetune_df.groupby(['backbone','frac_dtarget'])['metric'].agg(['mean', 'std']).reset_index()
display(mean_df)
mean_df["frac_dtarget"] = mean_df["frac_dtarget"].astype(str)

# Convert sample values to appropriate format
sample_mapping = {
    "1.0": "1",
    "5.0": "5", 
    "10.0": "10",
    "25.0": "25",
    "50.0": "50",
    "100.0": "100",
    "200.0": "200",
    "1000.0": "max"
}
mean_df["frac_dtarget"] = mean_df["frac_dtarget"].map(sample_mapping)

# Set enhanced style parameters
sns.set_style("whitegrid", rc={
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.titlesize": 14,
})

# Create figure with larger dimensions
plt.figure(figsize=(11, 7), layout='constrained')

# Create pointplot with standard deviation as error bars
ax = sns.pointplot(
    data=mean_df, 
    x='frac_dtarget', 
    y='mean', 
    hue='backbone',
    palette=custom_palette,
    hue_order=["RNN", "IMU Transformer", "ResNet-1D",'ResNet-SE-5', "CNN-PFF", "TS2Vec Encoder",'TS-TCC Encoder'],
    order=["1", "5", "10", "25", "50", "100", "200", "max"],
    errwidth=2.0,      # Thicker error bars
    capsize=0.15,      # Caps on error bars
    ci='sd'            # Use standard deviation for error bars
)

# Enhanced axis labels
plt.xlabel("Number of Samples", 
          fontsize=16,
          fontweight='bold',
          labelpad=12)
plt.ylabel("Balanced Accuracy", 
          fontsize=16,
          fontweight='bold',
          labelpad=12)

# Bold axis ticks with larger font
plt.xticks(rotation=45, 
          fontsize=13,
          fontweight='bold')
plt.yticks(fontsize=13,
          fontweight='bold')

# Y-axis formatting
plt.ylim(0.3, 1.0)
ax.set_yticklabels(['{:.0f}%'.format(y*100) for y in ax.get_yticks()])

# Enhanced grid
plt.grid(True, alpha=0.7)

# Upgraded legend
handles, labels = ax.get_legend_handles_labels()
legend = plt.legend(
    handles, 
    labels, 
    title="Backbone",
    title_fontsize=18,
    fontsize=16,
    bbox_to_anchor=(1, 1),
    loc='upper left',
    frameon=True,
    framealpha=1,
    edgecolor='black'
)

# Make legend title bold
legend.get_title().set_fontweight('bold')

# Save high-quality output
plt.savefig('finetune_rq42.png', 
           dpi=350,
           bbox_inches='tight', 
           transparent=True)

plt.show()

CNNs e Resnet bem mais significante em 5 e 10% dos dados